## Problem 4.14 — Strawberry Jam

Strawberries contain about **15 wt% solids** and **85 wt% water**. To make strawberry jam, crushed strawberries and sugar are mixed in a **45:55 mass ratio**, and the mixture is heated to evaporate water until the residue contains **one-third water by mass**.

**(a)** Draw and label a flowchart of this process.

**(b)** Do the degree-of-freedom analysis and show that the system has zero degrees of freedom (i.e., the number of unknown process variables equals the number of equations relating them). If you have too many unknowns, think about what you might have forgotten to do.

**(c)** Calculate how many pounds of strawberries are needed to make a pound of jam.

**(d)** Making a pound of jam is something you could accomplish in your own kitchen (or maybe even a dorm room). However, a typical manufacturing line for jam might produce **1500 lbm/h**. List technical and economic factors you would have to take into account as you scaled up this process from your kitchen to a commercial operation.

![problem4.14](../artifacts/strawberry_4_14_env.png)

4 variables unknown: $m_M, m_W, m_G, m_A$

2 components $\to$ water and solids = 2 independent material balances

1 independent relationship: $\frac{m_M}{m_A}=\frac{45}{55}=k$

1 degree of freedom

- Hypotheses:
  - no accumulation
  - no reaction inside the heater

material balance for water: $0.85m_M = m_W + 1/3m_G$

material balance for solids: $0.15m_M + m_A = 2/3 m_G$

we know that: $m_M = km_A$

$$0.85(km_A) = m_W + 1/3m_G$$

$$0.15(km_A) + m_A = 2/3 m_G$$

suppose we want to calculate the flows based on $m_G$, we can reorganize this system into:

$$0.85k m_A-m_W=1/3m_G$$

$$(0.15k+1)m_A=2/3 m_G$$

or, in linear system notation:

\begin{bmatrix}0.85k & 1 \\ (0.15k+1) & 0\end{bmatrix}\begin{bmatrix}m_A \\ m_W\end{bmatrix}=\begin{bmatrix}1/3m_G \\ 2/3m_G\end{bmatrix}

In [1]:
import numpy as np

def calculate_stream_based_on_jam_mass(
        m_jam: float,
        k: float = 45./55.,
        strawberry_water_content: float = 0.85,
        jam_water_content: float = 1/3
) -> dict:
    """
    performs the material balance of jam heater to calculate all streams

    Args:
        m_jam (float): desired mass of jam produced
        k (float, optional): strawberry to sugar ratio. Defaults to 45./55..
        strawberry_water_content (float, optional): strawberry water conten. Defaults to 0.85.
        jam_water_content (float, optional): jam water content. Defaults to 1/3.

    Returns:
        dict: material balance metadata
    """
    strawberry_solids = 1 - strawberry_water_content

    # build the linear system
    A = np.array([
        [strawberry_water_content * k, -1],
        [strawberry_solids * (k + 1), 0]
    ])

    b = np.array([
        jam_water_content * m_jam,
        (1 - jam_water_content) * m_jam
    ])

    # solve the linear system
    sugar_mass, water_mass = np.linalg.solve(A, b)

    strawberry_mass = sugar_mass * k

    return {
        "strawberry_mass": strawberry_mass,
        "sugar_mass": sugar_mass,
        "water_mass": water_mass,
        "jam_mass": m_jam
    }

In [3]:
# Example usage
desired_jam_mass = 1000  # kg
result = calculate_stream_based_on_jam_mass(desired_jam_mass)
print(f"To produce {desired_jam_mass} kg of jam, you need:")
print(f"- {result['strawberry_mass']:.2f} kg of strawberries")
print(f"- {result['sugar_mass']:.2f} kg of sugar")
print(f"- {result['water_mass']:.2f} kg of water evaporated")

To produce 1000 kg of jam, you need:
- 2000.00 kg of strawberries
- 2444.44 kg of sugar
- 1366.67 kg of water evaporated


## Problem 4.6 — Partial Evaporation of an Acetone–Water Mixture

A liquid mixture of acetone and water contains **35 mole% acetone**. The mixture is to be partially evaporated to produce a vapor that is **75 mole% acetone** and leave a residual liquid that is **18.7 mole% acetone**.

### (a)

Suppose the process is to be carried out **continuously and at steady state** with a feed rate of **10.0 kmol/h**. Let $\dot{n}_v$ and $\dot{n}_l$ be the flow rates of the vapor and liquid product streams, respectively.

Draw and label a process flowchart, then write and solve balances on **total moles** and on **acetone** to determine the values of $\dot{n}_v$ and $\dot{n}_l$.

For each balance, state which terms in the general balance equation

$$
\text{accumulation}
=
\text{input}
+
\text{generation}
-
\text{output}
-
\text{consumption}
$$

can be discarded and why.

### (b)

Now suppose the process is to be carried out in a **closed container** that initially contains **10.0 kmol** of the liquid mixture. Let $n_v$ and $n_l$ be the moles of final vapor and liquid phases, respectively.

Draw and label a process flowchart, then write and solve **integral balances** on total moles and on acetone.

For each balance, state which terms of the general balance equation can be discarded and why.

### (c)

Returning to the continuous process, suppose the vaporization unit is built and started and the product stream flow rates and compositions are measured.

The measured acetone content of the vapor stream is **75 mole% acetone**, and the product stream flow rates have the values calculated in Part (a). However, the liquid product stream is found to contain **22.3 mole% acetone**.

It is possible that there is an error in the measured composition of the liquid stream, but give **at least five other reasons** for the discrepancy.

*Think about assumptions made in obtaining the solution of Part (a).*

In [36]:
from scipy.optimize import least_squares

# store components properties
class Component:
    def __init__(self, name: str):
        self.name = name

# store stream properties
class Stream:
    def __init__(
            self,
            name: str,
            flow_rate: float, 
            flow_type: str, 
            components: list[Component],
            composition: dict[str, float],
            direction: str = 'input'
        ) -> None:
        """
        describes a stream in a process unit

        Args:
            name (str): name or tag of the stream
            flow_rate (float): stream flow rate
            flow_type (str): stream flow type (mass, molar, volumetric)
            components (list[Component]): list of components in the stream
            composition (dict[str, float]): composition of the stream
            direction (str, optional): direction of the stream. Defaults to 'input'.
        """
        self.name = name
        self.flow_rate = flow_rate
        self.flow_type = flow_type
        self.components = components
        self.composition = composition
        self.direction = direction

        # validate if len(components) == len(composition) - private
        self._validate_components_composition()

        # validate if composition values sum to 1 - private
        self._validate_composition_sum()

        # if one composition is None, calculate the complementary composition
        self.calculate_complementary_composition()

    def _validate_components_composition(self):
        if len(self.components) != len(self.composition):
            raise ValueError("Number of components must match the number of composition values.")

    def _validate_composition_sum(self):
        if all(value is not None for value in self.composition.values()):
            if not np.isclose(sum(self.composition.values()), 1.0):
                raise ValueError("Composition values must sum to 1.")

    # if one composition is None, calculate the complementary composition
    def calculate_complementary_composition(self):
        if None in self.composition.values():
            known_sum = sum(value for value in self.composition.values() if value is not None)
            missing_component = next(key for key, value in self.composition.items() if value is None)
            self.composition[missing_component] = 1.0 - known_sum

    # validate if all n -1 composition values are provided, warning if not
    def validate_composition_values(self):
        if list(self.composition.values()).count(None) > 1:
            print("Warning: More than one composition value is missing. Cannot calculate complementary composition.")

class ProcessUnit:
    def __init__(self, name: str, input_streams: list[Stream], output_streams: list[Stream]) -> None:
        """
        describes a process unit

        Args:
            name (str): name or tag of the process unit
            input_streams (list[Stream]): list of input streams
            output_streams (list[Stream]): list of output streams
        """
        self.name = name
        self.tol = 1e-6
        self.input_streams = input_streams
        self.output_streams = output_streams

        self._validate_stream_components()

        # method to calculate the number of independent material balances
        self.calculate_independent_material_balances()

        # method to calculate the number of unknowns
        self.calculate_unknowns()

        # suggest missing information if the system is not solvable
        self.suggest_missing_information()

    # validate if all input and output streams have the same components - private
    def _validate_stream_components(self):
        input_components = {comp.name for stream in self.input_streams for comp in stream.components}
        output_components = {comp.name for stream in self.output_streams for comp in stream.components}
        if input_components != output_components:
            raise ValueError("Input and output streams must have the same components.")

    # calculate the number of independent material balances = number of components
    def calculate_independent_material_balances(self):
        self.independent_material_balances = max(len(stream.components) for stream in self.input_streams + self.output_streams)
        print(f"Process unit '{self.name}' has {self.independent_material_balances} independent material balances.")

    # calculate the number of unknowns: any None in streams
    def calculate_unknowns(self):
        self.unknowns = sum(
            int(stream.flow_rate is None)
            + list(stream.composition.values()).count(None)
            for stream in self.input_streams + self.output_streams
        )
        print(
            f"Process unit '{self.name}' has {self.unknowns} "
            "unknowns in the material balances."
        )

    # method to check if the system is solvable
    def is_solvable(self):
        if self.unknowns > self.independent_material_balances:
            print(f"Process unit '{self.name}' is not solvable: {self.unknowns} unknowns > {self.independent_material_balances} independent material balances.")
            return False
        else:
            print(f"Process unit '{self.name}' is solvable: {self.unknowns} unknowns <= {self.independent_material_balances} independent material balances.")
            return True

    # method to suggest what is still missing to make the system solvable
    def suggest_missing_information(self):
        if self.is_solvable():
            print(f"Process unit '{self.name}' is already solvable. No additional information needed.")
            return

        missing_info = []

        for stream in self.input_streams + self.output_streams:
            if stream.flow_rate is None:
                missing_info.append(
                    f"Stream '{stream.name}': Missing flow rate."
                )

            for comp_name, comp_value in stream.composition.items():
                if comp_value is None:
                    missing_info.append(
                        f"Stream '{stream.name}': "
                        f"Missing composition for component '{comp_name}'."
                    )

        if missing_info:
            print(
                "To make the system solvable, consider providing the "
                "following missing information:"
            )
            for info in missing_info:
                print(info)
        else:
            print(
                "No specific missing information identified, but the system "
                "is still not solvable."
            )

    # collect (stream, kind, key) references for every unknown value still set to None
    def _collect_unknowns(self):
        unknown_refs = []
        for stream in self.input_streams + self.output_streams:
            if stream.flow_rate is None:
                unknown_refs.append((stream, "flow_rate", None))
            for comp_name, comp_value in stream.composition.items():
                if comp_value is None:
                    unknown_refs.append((stream, "composition", comp_name))
        return unknown_refs

    # write a vector of candidate values back onto the streams they belong to
    def _apply_unknowns(self, unknown_refs, values):
        for (stream, kind, key), value in zip(unknown_refs, values):
            if kind == "flow_rate":
                stream.flow_rate = value
            else:
                stream.composition[key] = value

    # residual of each component balance: total input - total output, should be 0 at the solution
    def _material_balance_residuals(self, values, unknown_refs, component_names):
        self._apply_unknowns(unknown_refs, values)

        residuals = []
        for comp_name in component_names:
            input_total = sum(
                stream.flow_rate * stream.composition[comp_name]
                for stream in self.input_streams
            )
            output_total = sum(
                stream.flow_rate * stream.composition[comp_name]
                for stream in self.output_streams
            )
            residuals.append(input_total - output_total)

        return residuals

    def print_report(self):
        """Print stream flow details and component and overall mass balance checks."""
        streams = self.input_streams + self.output_streams
        component_names = sorted({
            component.name for stream in streams for component in stream.components
        })

        print(f"\nProcess unit report: {self.name}")
        print(f"Mass balance tolerance: {self.tol:.1e}")

        for label, grouped_streams in (
            ("Input streams", self.input_streams),
            ("Output streams", self.output_streams),
        ):
            print(f"\n{label}:")
            if not grouped_streams:
                print("  (none)")
                continue

            for stream in grouped_streams:
                flow = "unknown" if stream.flow_rate is None else f"{stream.flow_rate:.6g} {stream.flow_type}"
                print(f"  {stream.name}: total flow = {flow}")
                for component_name in component_names:
                    fraction = stream.composition.get(component_name)
                    fraction_text = "unknown" if fraction is None else f"{fraction:.6g}"
                    if stream.flow_rate is None or fraction is None:
                        component_flow_text = "unknown"
                    else:
                        component_flow_text = f"{stream.flow_rate * fraction:.6g} {stream.flow_type}"
                    print(
                        f"    {component_name}: fraction = {fraction_text}, "
                        f"component flow = {component_flow_text}"
                    )

        print("\nMass balance checks:")
        for component_name in component_names:
            input_flow = sum(
                stream.flow_rate * stream.composition[component_name]
                for stream in self.input_streams
            ) if all(
                stream.flow_rate is not None and stream.composition.get(component_name) is not None
                for stream in self.input_streams
            ) else None
            output_flow = sum(
                stream.flow_rate * stream.composition[component_name]
                for stream in self.output_streams
            ) if all(
                stream.flow_rate is not None and stream.composition.get(component_name) is not None
                for stream in self.output_streams
            ) else None

            if input_flow is None or output_flow is None:
                print(f"  {component_name}: unavailable (stream flow or composition is unknown)")
                continue

            residual = input_flow - output_flow
            status = "PASS" if abs(residual) <= self.tol else "FAIL"
            print(
                f"  {component_name}: input = {input_flow:.6g}, "
                f"output = {output_flow:.6g}, residual = {residual:.6g} [{status}]"
            )

        if any(stream.flow_rate is None for stream in streams):
            print("  Overall: unavailable (one or more stream flow rates are unknown)")
        else:
            total_input = sum(stream.flow_rate for stream in self.input_streams)
            total_output = sum(stream.flow_rate for stream in self.output_streams)
            residual = total_input - total_output
            status = "PASS" if abs(residual) <= self.tol else "FAIL"
            print(
                f"  Overall: input = {total_input:.6g}, output = {total_output:.6g}, "
                f"residual = {residual:.6g} [{status}]"
            )

    def solve_material_balances(self):
        if not self.is_solvable():
            print(f"Process unit '{self.name}' cannot be solved due to insufficient information.")
            return

        print(f"Solving material balances for process unit '{self.name}'...")

        unknown_refs = self._collect_unknowns()

        if not unknown_refs:
            print(f"Process unit '{self.name}' has no unknowns left to solve for.")
            return

        component_names = sorted({
            comp.name for stream in self.input_streams + self.output_streams for comp in stream.components
        })

        # initial guesses: total known input flow (or 1.0) for flow rates, even split for compositions
        known_input_flow = sum(
            stream.flow_rate for stream in self.input_streams if stream.flow_rate is not None
        ) or 1.0
        x0 = [
            known_input_flow if kind == "flow_rate" else 1.0 / len(stream.components)
            for stream, kind, _ in unknown_refs
        ]

        # solve for sum inputs = sum of all input stream flow rates, varying the unknowns (scipy)
        result = least_squares(
            self._material_balance_residuals,
            x0=x0,
            args=(unknown_refs, component_names),
            bounds=(0.0, np.inf),  # flow rates and mole/mass fractions are non-negative
        )

        if not result.success:
            print(f"Process unit '{self.name}' could not be solved: {result.message}")
            return

        self._apply_unknowns(unknown_refs, result.x)

        print(f"Solved material balances for process unit '{self.name}':")
        for stream in self.input_streams + self.output_streams:
            composition_str = ", ".join(f"{k}={v:.4f}" for k, v in stream.composition.items())
            print(f"  {stream.name}: flow_rate={stream.flow_rate:.4f} ({composition_str})")

        return result.x

In [37]:
# create components
acetone = Component("Acetone")
water = Component("Water")

# input stream
input_stream = Stream(
    name="n",
    flow_rate=10,  # kmol/h
    flow_type="mole",  # example flow type
    components=[acetone, water],
    composition={"Acetone": 0.35, "Water": None},
    direction="input"
)
vapor_stream = Stream(
    name="vapor",
    flow_rate=None,  # kmol/h
    flow_type="mole",  # example flow type
    components=[acetone, water],
    composition={"Acetone": 0.75, "Water": None},
    direction="output"
)
liquid_stream = Stream(
    name="liquid",
    flow_rate=None,  # kmol/h
    flow_type="mole",  # example flow type
    components=[acetone, water],
    composition={"Acetone": 0.187, "Water": None},
    direction="output"
)

# create process unit - evaporator
evaporator = ProcessUnit(
    name="Evaporator",
    input_streams=[input_stream],
    output_streams=[vapor_stream, liquid_stream]
)

# solve the material balances for the evaporator
evaporator.solve_material_balances()

Process unit 'Evaporator' has 2 independent material balances.
Process unit 'Evaporator' has 2 unknowns in the material balances.
Process unit 'Evaporator' is solvable: 2 unknowns <= 2 independent material balances.
Process unit 'Evaporator' is already solvable. No additional information needed.
Process unit 'Evaporator' is solvable: 2 unknowns <= 2 independent material balances.
Solving material balances for process unit 'Evaporator'...
Solved material balances for process unit 'Evaporator':
  n: flow_rate=10.0000 (Acetone=0.3500, Water=0.6500)
  vapor: flow_rate=2.8952 (Acetone=0.7500, Water=0.2500)
  liquid: flow_rate=7.1048 (Acetone=0.1870, Water=0.8130)


array([2.89520426, 7.10479574])

In [38]:
evaporator.solve_material_balances()

Process unit 'Evaporator' is solvable: 2 unknowns <= 2 independent material balances.
Solving material balances for process unit 'Evaporator'...
Process unit 'Evaporator' has no unknowns left to solve for.


In [39]:
evaporator.print_report()


Process unit report: Evaporator
Mass balance tolerance: 1.0e-06

Input streams:
  n: total flow = 10 mole
    Acetone: fraction = 0.35, component flow = 3.5 mole
    Water: fraction = 0.65, component flow = 6.5 mole

Output streams:
  vapor: total flow = 2.8952 mole
    Acetone: fraction = 0.75, component flow = 2.1714 mole
    Water: fraction = 0.25, component flow = 0.723801 mole
  liquid: total flow = 7.1048 mole
    Acetone: fraction = 0.187, component flow = 1.3286 mole
    Water: fraction = 0.813, component flow = 5.7762 mole

Mass balance checks:
  Acetone: input = 3.5, output = 3.5, residual = -8.88178e-16 [PASS]
  Water: input = 6.5, output = 6.5, residual = 0 [PASS]
  Overall: input = 10, output = 10, residual = 0 [PASS]
